# E5a v3 — redundancy robustness

Adds 50 correlated copies after the 5,000-gene variance prefilter, then compares raw and copy-collapsed Nogueira stability. Default is non-wrapper methods; run wrappers separately only if required.

In [ ]:
from pathlib import Path
import os,sys,subprocess,shutil,json,warnings
INPUT=Path('/kaggle/input/experiment-data-v3'); REPO=Path('/kaggle/working/RNA-Seq-FS-Survey'); WORK=Path('/kaggle/working/kltn_data')
COHORTS=['tcga_luad_lusc_primary_tumor','gse10245','gse30219','gse37745','gse42127']
METHODS=['f_test','mutual_info','mrmr','relieff','l1_lr','elastic_net','rf_importance','svm_rfe'] # add ga,bpso only in a dedicated wrapper run
RHO=[0.5,0.8,0.95]; N_SUBSAMPLES=30; K_GRID=[1,2,3,4,5,10,25,50]; FRACTION=.7; N_COPIES=50
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch','survey-v2-code','--single-branch','https://github.com/datachain-uit/RNA-Seq-FS-Survey.git',str(REPO)],check=True)
try: import skrebate
except ModuleNotFoundError: subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','skrebate==0.8.4'],check=True)
WORK.mkdir(parents=True,exist_ok=True)
for name in ['experiment-data-perm','experiment-data-geo-target','split_manifests']:
 src,tgt=INPUT/name,WORK/name; assert src.is_dir(),src
 if not tgt.exists(): tgt.symlink_to(src,target_is_directory=True)
warnings.filterwarnings('ignore',message=r'Features .* are constant\.',category=UserWarning); warnings.filterwarnings('ignore',message=r'invalid value encountered in divide',category=RuntimeWarning)
os.chdir(REPO);sys.path.insert(0,str(REPO))

In [ ]:
import numpy as np,pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.data import load_dataset,variance_prefilter,stratified_subsample,add_correlated_copies
from survey_v2.methods import select
from survey_v2.stability import nogueira,selections_to_matrix
ROOT=WORK/'results'/'survey_v3'/'E5a_redundancy'
def save(df,p): p.parent.mkdir(parents=True,exist_ok=True); t=p.with_suffix('.tmp');df.to_csv(t,index=False);t.replace(p)
for cohort in COHORTS:
 ds=load_dataset(WORK,cohort); out=ROOT/cohort
 for rho in RHO:
  for s in range(N_SUBSAMPLES):
   idx=stratified_subsample(ds.y,FRACTION,np.random.default_rng([C.dataset_seed_id(cohort),5,int(rho*100),s])); pre=variance_prefilter(ds.X[idx],5000); xb=ds.X[idx][:,pre]; gb=ds.genes[pre]
   xa,ga,_,_=add_correlated_copies(xb,gb,rho,N_COPIES,np.random.default_rng([C.dataset_seed_id(cohort),51,int(rho*100),s]))
   for mid,method in enumerate(METHODS):
    p=out/'checkpoints'/f'rho{int(rho*100)}_s{s}_{method}.csv'
    if p.is_file(): continue
    with threadpool_limits(1): panels=select(method,xa,ds.y[idx],K_GRID,seed=C.seed_for(C.dataset_seed_id(cohort),5,int(rho*100),s,mid),n_jobs=1).panels
    rows=[{'cohort':cohort,'rho':rho,'subsample':s,'method':method,'k':k,'rank':rank,'gene_raw':ga[col],'gene_merged':ga[col].split('__copy_rho')[0]} for k,panel in panels.items() for rank,col in enumerate(panel,1)]
    save(pd.DataFrame(rows),p); print(cohort,rho,s,method,flush=True)
 files=sorted((out/'checkpoints').glob('*.csv')); expected=len(RHO)*N_SUBSAMPLES*len(METHODS); assert len(files)==expected,(cohort,len(files),expected)
 genes=pd.concat([pd.read_csv(p) for p in files],ignore_index=True); save(genes,out/'e5a_selected_genes.csv')
 rows=[]
 for (rho,method,k),g in genes.groupby(['rho','method','k']):
  raw=[x.gene_raw.tolist() for _,x in g.groupby('subsample')]; merged=[x.gene_merged.tolist() for _,x in g.groupby('subsample')]
  rows.append({'cohort':cohort,'fraction':FRACTION,'rho':rho,'method':method,'k':k,**{f'raw_{a}':b for a,b in nogueira(selections_to_matrix(raw,np.unique(genes.gene_raw))).items()},**{f'merged_{a}':b for a,b in nogueira(selections_to_matrix(merged,ds.genes)).items()}})
 save(pd.DataFrame(rows),out/'e5a_redundancy.csv')
 (out/'config.json').write_text(json.dumps({'experiment':'E5a_redundancy_v3','fraction':FRACTION,'rho':RHO,'n_copies':N_COPIES,'n_subsamples':N_SUBSAMPLES,'methods':METHODS,'k_grid':K_GRID},indent=2))
shutil.make_archive('/kaggle/working/e5a_redundancy_v3','zip',WORK,'results/survey_v3/E5a_redundancy');print('DONE')